In [21]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import seaborn as sns
import os

from glob import glob
import librosa, librosa.display
import IPython.display as ipd

import random
import shutil

dir_folder = os.listdir('../data/Data/')


Sampling Strata

In [22]:
def daftar_subfolder() :
    for folder in dir_folder:
        data_path = '../data/Data/' + folder + '/asr_javanese/data/'
        subfolders = os.listdir(data_path)
        print(f"Folder '{folder}' subfolders: {sorted(subfolders)}")

In [23]:
daftar_subfolder()

Folder '0' subfolders: ['00', '01', '02', '03', '04', '05', '06', '07', '08', '09', '0a', '0b', '0c', '0d', '0e', '0f']
Folder '1' subfolders: ['10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '1a', '1b', '1c', '1d', '1e', '1f']
Folder '10' subfolders: ['a0', 'a1', 'a2', 'a3', 'a4', 'a5', 'a6', 'a7', 'a8', 'a9', 'aa', 'ab', 'ac', 'ad', 'ae', 'af']
Folder '11' subfolders: ['b0', 'b1', 'b2', 'b3', 'b4', 'b5', 'b6', 'b7', 'b8', 'b9', 'ba', 'bb', 'bc', 'bd', 'be', 'bf']
Folder '12' subfolders: ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9', 'ca', 'cb', 'cc', 'cd', 'ce', 'cf']
Folder '13' subfolders: ['d0', 'd1', 'd2', 'd3', 'd4', 'd5', 'd6', 'd7', 'd8', 'd9', 'da', 'db', 'dc', 'dd', 'de', 'df']
Folder '14' subfolders: ['e0', 'e1', 'e2', 'e3', 'e4', 'e5', 'e6', 'e7', 'e8', 'e9', 'ea', 'eb', 'ec', 'ed', 'ee', 'ef']
Folder '15' subfolders: ['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'fa', 'fb', 'fc', 'fd', 'fe', 'ff']
Folder '2' subfolders: ['20', '21'

In [24]:
def konversi_subfolder():
    all_data = []

    for folder in dir_folder:
        data_path = '../data/Data/' + folder + '/asr_javanese/data/'
        
        # Konversi nama folder (0-15) ke hex (O-f)
        hex_prefix = format(int(folder), 'x')  
        
        folder_data = []
        
        for i in range(16):
            subfolder = hex_prefix + format(i, 'x')
            subpath = data_path + subfolder + '/*.flac'
            files = glob(subpath)
            if files:
                folder_data.append(files)
        
        if folder_data:
            all_data.append(folder_data)
    print('Konversi Nama Subfolder.')
    # print("Jumlah Folder Utama :",len(all_data))        #  16 folder utama
    # print("Jumlah Sub Folder   :",len(all_data[0]))     #  16 sub folder
    # print("Jumlah File pada folder", (folder),"pada sub folder", (subfolder),"adalah",len(all_data[0][0]))  # jumlah file pada folder 0 subfolder 00
    # print("Contoh Path data ", folder_data[0][0])
    
    return all_data

In [ ]:
konversi_subfolder()

In [26]:
def stratified_Sampling(random_seed, persen_sampling):
    
    random.seed(random_seed)
    sampled_data = []
    total_all = 0
    total_sample = 0
    print('Stratified Sampling.')
    for folder_idx, folder in enumerate(dir_folder):
        data_path = '../data/Data/' + folder + '/asr_javanese/data/'
        hex_prefix = format(int(folder), 'x')
        
        folder_sampled = []
        
        for i in range(16):
            subfolder = hex_prefix + format(i, 'x')
            subpath = data_path + subfolder + '/*.flac'
            files = glob(subpath)
            
            if files:
                # Ambil 10% dari tiap subfolder
                n_sample = max(1, int(len(files) * persen_sampling))
                sampled_files = random.sample(files, n_sample)
                
                total_all+= len(files)
                total_sample+=n_sample
                # Buat folder tujuan baru
                dest_dir = f'../data/sampled/{folder}/{subfolder}'
                os.makedirs(dest_dir, exist_ok=True)
                
                # Copy file ke folder tujuan
                for f in sampled_files:
                    shutil.copy(f, dest_dir)
                
                folder_sampled.append(sampled_files)
                # print(f"Folder {folder}/{subfolder}: {len(files)} file -> sampled {n_sample}")
        
        sampled_data.append(folder_sampled)
    print('Total Keseluruhan :', total_all)
    print('Total Sampling    :', total_sample)
    return sampled_data


In [ ]:
stratified_Sampling(42,  0.1)

In [28]:
def dataFrame(all_data, sampled_data):
    rows = []

    for folder_idx, folder in enumerate(dir_folder):
        hex_prefix = format(int(folder), 'x')
        
        for i in range(16):
            subfolder = hex_prefix + format(i, 'x')
            
            # Data asli
            all_files = all_data[folder_idx][i] if i < len(all_data[folder_idx]) else []
            # Data sampled
            sampled_files = sampled_data[folder_idx][i] if i < len(sampled_data[folder_idx]) else []
            
            rows.append({
                'folder'        : folder,
                'subfolder'     : subfolder,
                'total_file'    : len(all_files),
                'total_sampled' : len(sampled_files),
                'persen'        : round(len(sampled_files) / len(all_files) * 100, 2) if all_files else 0
            })

    df = pd.DataFrame(rows)
    total_row = pd.DataFrame([{
        'folder'        : 'TOTAL',
        'subfolder'     : len(df['subfolder']),
        'total_file'    : df['total_file'].sum(),
        'total_sampled' : df['total_sampled'].sum(),
        'persen'        : round(df['total_sampled'].sum() / df['total_file'].sum() * 100, 2)
    }])

    df = pd.concat([df, total_row], ignore_index=True)

    print(df.to_string(index=False))
    return all_files

In [ ]:
all_data = konversi_subfolder()
sampled_data = stratified_Sampling(42, 0.1)
all_files=dataFrame(all_data, sampled_data)

In [ ]:
df=pd.read_csv('../data/Data/15/asr_javanese/utt_spk_text.tsv', sep='\t')
header = ['file_name','id','sentences']
df.columns=header
df.head()

In [ ]:
sampled_root = '../data/sampled'

sampled_rows = []
for folder in os.listdir(sampled_root):
    folder_path = os.path.join(sampled_root, folder)
    if not os.path.isdir(folder_path):
        continue
    for subfolder in os.listdir(folder_path):
        subfolder_path = os.path.join(folder_path, subfolder)
        if not os.path.isdir(subfolder_path):
            continue
        for fname in os.listdir(subfolder_path):
            file_stem = os.path.splitext(fname)[0]
            sampled_rows.append({
                'folder': folder,
                'subfolder': subfolder,
                'file_name': file_stem
            })

sampled_df = pd.DataFrame(sampled_rows)

df['file_name_stem'] = df['file_name'].apply(lambda x: os.path.splitext(os.path.basename(x))[0])

matched_df = sampled_df.merge(df[['file_name_stem', 'id', 'sentences']],
                               left_on='file_name',
                               right_on='file_name_stem',
                               how='left')

print(f"Total File Sampling: {len(matched_df)}")
print(f"Cocok terhadap id  : {matched_df['id'].notna().sum()}")
print(f"Tidak cocok        : {matched_df['id'].isna().sum()}")

In [ ]:
matched_df

In [33]:
all_paths = [
    os.path.join(root_dir, file)
    for root_dir, dirs, files in os.walk(sampled_root)
    for file in files
]

In [34]:
matched_df.insert(loc=6, column='Path_Data',value = all_paths)

In [35]:
matched_df2 =matched_df.drop(columns=['file_name','file_name_stem','id'])

In [ ]:
matched_df2

In [37]:
matched_df2.to_csv("dataClean.csv",index=False)